# 最高到達accuracyを取得する

In [1]:
from datetime import date

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import json
import re
import sys
from collections import defaultdict

project_root = os.path.join(os.getcwd(), "..")
result_dir = os.path.join(project_root, "results")


def load_file_and_extract_max_accs(date, lr, target_concept_config_name, model_size, result_dir):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""
    # if target_concept_config_type != '':
    #     target_concept_config_type = '-' + target_concept_config_type
    # dirname_pattern1 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_target_concepts_initvecwith(.*)"
    # dirname_pattern2 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-seed(\\d+)_target_concepts_initvecwith(.*)"

    target_concept_config_name = target_concept_config_name.replace('.json', '')
    dirname_pattern1 = f"gemma-3-{model_size}B-lr{lr}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    dirname_pattern2 = f"gemma-3-{model_size}B-lr{lr}-{date}-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    initVecType_to_seed_to_maxacc = defaultdict(dict)
    initVecType_to_seed_to_epoch_at_maxacc = defaultdict(dict)
    for dirname in os.listdir(result_dir):

        if re.match(dirname_pattern1, dirname):
            match_groups = re.search(dirname_pattern1, dirname)
            layer_idx = int(match_groups.group(1))
            seed = int(match_groups.group(2))
            initVecType = f"{match_groups.group(3)}_layer{layer_idx}"
        elif re.match(dirname_pattern2, dirname):
            match_groups = re.search(dirname_pattern2, dirname)
            print(match_groups)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
        else:
            if date in dirname:
                # print(dirname)
                pass
            continue  # ディレクトリ名がどちらのパターンにもマッチしない場合はスキップ

        # print(f"Processing directory: {dirname}, initVecType: {initVecType}, seed: {seed}")
        
        if not os.path.exists(os.path.join(result_dir, dirname, 'logit_score_summary.json')):
            # print(f"Warning: logit_score_summary.json not found in {dirname}. Skipping.")
            continue
        with open(os.path.join(result_dir, dirname, 'logit_score_summary.json'), "r") as f:
            concept_to_epoch_to_scores = json.load(f)

        # 各conceptについて、回したepochs中の最高到達accuracyとそのepochを抽出
        max_accs = []
        epochs_at_maxacc = []
        for epoch_to_scores in concept_to_epoch_to_scores.values():
            max_accuracy = 0
            max_epoch_at_maxacc = 0
            for epoch, scores in epoch_to_scores.items():
                # print(f"Epoch: {epoch}, Accuracy: {scores['accuracy']}")
                if scores['accuracy'] > max_accuracy:
                    max_accuracy = scores['accuracy']
                    max_epoch_at_maxacc = epoch
            max_accs.append(max_accuracy)
            epochs_at_maxacc.append(max_epoch_at_maxacc)
        # print(f"Max Accuracies: {max_accs}")
        # print(f"Epochs at Max Accuracies: {epochs_at_maxacc}")
        initVecType_to_seed_to_maxacc[initVecType][seed] = max_accs
        initVecType_to_seed_to_epoch_at_maxacc[initVecType][seed] = epochs_at_maxacc
    return initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc


## 12B

In [2]:

date = '20260423'
model_size = '12'
lr = 0.003

target_concept_config_name = 'target_concepts_mini_13.json'


In [3]:

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

seeds = set([seed for seed_to_maxacc in initVecType_to_seed_to_maxacc.values() for seed in seed_to_maxacc.keys()])
initVecType_lst = sorted(list(initVecType_to_seed_to_maxacc.keys()))

for seed in seeds:
    print(f"Seed: {seed}")
    for initVecType in initVecType_lst:
        max_accs = initVecType_to_seed_to_maxacc[initVecType].get(seed, None)
        if max_accs:
            mean_max_acc = np.mean(max_accs)
            std_max_acc = np.std(max_accs)
            print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")

print('\n' + '-' * 100 + '\n')

for seed in seeds:
    print(f"Seed: {seed}")
    for initVecType in initVecType_lst:
        epochs_at_maxacc = initVecType_to_seed_to_epoch_at_maxacc[initVecType].get(seed, None)

        if epochs_at_maxacc:
            mean_epoch_at_maxacc = np.mean([int(e) for e in epochs_at_maxacc])
            std_epoch_at_maxacc = np.std([int(e) for e in epochs_at_maxacc])
        print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Epoch number at Max Accuracy: {mean_epoch_at_maxacc:.2f}, Std: {std_epoch_at_maxacc:.2f}")


Seed: 0
	InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7496, Std: 0.1168
	InitVecType: nearCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7620, Std: 0.1453
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7485, Std: 0.1247
Seed: 1
	InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.7447, Std: 0.1208
	InitVecType: nearCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.7565, Std: 0.1459
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.7727, Std: 0.1211
Seed: 2
	InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Mean Max Accuracy: 0.7811, Std: 0.1372
	InitVecType: nearCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Mean Max Accuracy: 0.7516, Std: 0.1353
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Mean Max Accuracy: 0.7494

## epoch4固定でaccuracyを計算する

In [4]:
sys.path.append(project_root)

from utils.gemma_train_and_test_utils import calculate_metrics

def get_concept_to_epoch_score(concept_result_dir, epoch):
    concept_to_results = {}
    for dirname in os.listdir(concept_result_dir):
        if dirname.endswith(".json"):
            continue
        concept = dirname

        # このconceptに対するepochの結果を読み込む
        logit_score_path = os.path.join(concept_result_dir, dirname, f"logit-scored_epoch{epoch}.json")
        
        if not os.path.exists(logit_score_path):
            # raise FileNotFoundError(f"Logit score file not found: {logit_score_path}")
            # print(f"Logit score file not found: {logit_score_path}. Skipping this concept.")
            continue
            
        with open(logit_score_path, 'r') as f:
            results = json.load(f)
        y_true_lst = [res['label'] for res in results]
        y_pred_lst = [res['pred'] for res in results]

        # score計算
        metrics = calculate_metrics(y_pred_lst, y_true_lst)

        concept_to_results[concept] = metrics

        # print(f"  Epoch {epoch}: Accuracy: {metrics['accuracy']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['F1']:.4f}")
    
    return concept_to_results





def load_file_and_calc_epoch_accs(date, epoch, lr, target_concept_config_name, model_size, result_dir, print_flag=False):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""

    target_concept_config_name = target_concept_config_name.replace('.json', '')
    dirname_pattern1 = f"gemma-3-{model_size}B-lr{lr}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    dirname_pattern2 = f"gemma-3-{model_size}B-lr{lr}-{date}-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"

    initvectype_to_seed_to_acc = defaultdict(dict)
    for dirname in os.listdir(result_dir):

        if re.match(dirname_pattern1, dirname):
            match_groups = re.search(dirname_pattern1, dirname)
            layer_idx = int(match_groups.group(1))
            seed = int(match_groups.group(2))
            initVecType = f"{match_groups.group(3)}_layer{layer_idx}"
        elif re.match(dirname_pattern2, dirname):
            match_groups = re.search(dirname_pattern2, dirname)
            print(match_groups)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
        else:
            if date in dirname:
                pass
            continue  # ディレクトリ名がどちらのパターンにもマッチしない場合はスキップ

        concept_to_results = get_concept_to_epoch_score(os.path.join(result_dir, dirname), epoch)
        if len(concept_to_results) == 0:
            # print(f"No valid concepts with logit score files found for directory: {dirname}. Skipping.")
            continue
        sum_acc = 0
        for concept, score in concept_to_results.items():
            acc = score['accuracy']
            sum_acc += acc

        avg_acc = sum_acc / len(concept_to_results)
        initvectype_to_seed_to_acc[initVecType][seed] = avg_acc
        if print_flag:
            print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {avg_acc:.4f}")

    return initvectype_to_seed_to_acc


In [7]:
layer = 12
seed_num = 10

for epoch in range(1, 5):   # train時にrange(maxEpoch)としていたせいでmaxEpoch=5でもepoch4までしか回していなかった。さらにmaxEpoch時点のモデルとして最終ループepoch4の結果が保存されてしまっていた。2026/04/20の結果まで。なので1引いて4epochまでの結果のみ読み込む
    print(f"Epoch: {epoch}")


    date = '20260420'
    initvectype_to_seed_to_acc = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)


    # ** Cat **
    initVecType = f'CatCent_by_WikiSummaryRepeatHSMixed_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            # print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
            pass
    print(acc_lst)
        
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    print()
    cat_acc_lst = acc_lst


    # ** otherCat **
    initVecType = f'otherCatCent_by_WikiSummaryRepeatHSMixed_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            # print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
            pass
    print(acc_lst)
    
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    print()
    othercat_acc_lst = acc_lst

    

    date = '20260423'
    initvectype_to_seed_to_acc = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)
    # ** nearCat **
    initVecType = f'nearCatCent_by_WikiSummaryRepeatHSMixed_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num): 
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            # print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
            pass
    print(acc_lst)
        
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    print()
    nearcat_acc_lst = acc_lst



    date = '20260418'
    initvectype_to_seed_to_acc = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)
    # ** norm_rand_vocab **
    initVecType = f'norm_rand_vocab_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)

    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    print()
    norm_rand_vocab_acc_lst = acc_lst



    # ** zero **
    initVecType = f'zero_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)

    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    zero_acc_lst = acc_lst

    print('\n==============================\n==============================\n')
    

Epoch: 1
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12
[0.6690990322246422, 0.6398637874499886, 0.664255977360044, 0.6272054061623815, 0.606414630393238, 0.6209320339782075, 0.5983449587136248, 0.6481432107785507, 0.6385762285706841, 0.6072072582273516]
accuracy at epoch 1: 0.6320 ± 0.0231

InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12
[0.5884485811423602, 0.6362810496088357, 0.6111544279520774, 0.6244642612373448, 0.6224738603129605, 0.632848353670089, 0.61891756486068, 0.5778030388733915, 0.6306407810858645, 0.6243556206340196]
accuracy at epoch 1: 0.6167 ± 0.0183

InitVecType: nearCatCent_by_WikiSummaryRepeatHSMixed_layer12
[0.6193067104976129, 0.6397094731565277, 0.6355212834976739, 0.5958875525636456, 0.6108715357161967, 0.6158449461364198, 0.6509010363807476, 0.6008354002198193, 0.6140420962741856, 0.6543327093142346]
accuracy at epoch 1: 0.6237 ± 0.0193

InitVecType: norm_rand_vocab_layer12
[0.5877783877011918, 0.6125625581591442, 0.6243171282314388, 0